# 10 — Engagement Analytics
Uses Employee_Performance_Dataset.csv (5000-person population). Department-level aggregations only — no per-employee joins to attrition data.

In [1]:

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'
ee = pd.read_csv(f'{PROC}/engagement_processed.csv')
print(f"Engagement dataset: {ee.shape}")
print(f"Columns: {list(ee.columns)}")
print(f"\nSample:\n{ee.head(3)}")


Engagement dataset: (5000, 13)
Columns: ['Employee ID', 'Name', 'Department', 'Job Role', 'Performance Score', 'KPI Score', 'Attendance (%)', 'Peer Rating', 'Task Completion (%)', 'Work Hours Logged', 'Manager Feedback', 'Training Hours', 'Promotion Eligibility']

Sample:
   Employee ID            Name Department             Job Role  \
0       376063  Manikya Badami      Sales      Sales Executive   
1       810690      Zara Mahal  Marketing  Marketing Executive   
2       520956       Sana Gera    Finance           Accountant   

   Performance Score  KPI Score  Attendance (%)  Peer Rating  \
0                 61      91.03           94.40          4.5   
1                 91      83.36           78.45          4.7   
2                100      65.39           99.50          4.4   

   Task Completion (%)  Work Hours Logged  Manager Feedback  Training Hours  \
0                78.19                 42               3.7              18   
1                71.39                 46      

In [2]:

# ── Department-level aggregation ──
numeric_cols = ['Performance Score', 'KPI Score', 'Attendance (%)',
                'Peer Rating', 'Task Completion (%)', 'Work Hours Logged', 'Training Hours']
numeric_cols = [c for c in numeric_cols if c in ee.columns]

dept_agg = ee.groupby('Department')[numeric_cols].agg(['mean', 'std', 'count']).round(3)
dept_agg.columns = ['_'.join(c) for c in dept_agg.columns]
dept_agg = dept_agg.reset_index()
print("=== Department-Level Aggregations ===")
print(dept_agg.to_string())


=== Department-Level Aggregations ===
  Department  Performance Score_mean  Performance Score_std  Performance Score_count  KPI Score_mean  KPI Score_std  KPI Score_count  Attendance (%)_mean  Attendance (%)_std  Attendance (%)_count  Peer Rating_mean  Peer Rating_std  Peer Rating_count  Task Completion (%)_mean  Task Completion (%)_std  Task Completion (%)_count  Work Hours Logged_mean  Work Hours Logged_std  Work Hours Logged_count  Training Hours_mean  Training Hours_std  Training Hours_count
0    Finance                  75.158                 14.753                     1016          77.393         10.191             1016               87.550               7.162                  1016             4.011            0.560               1016                    84.939                    8.892                       1016                  45.018                  6.022                     1016               15.474               8.921                  1016
1         Hr                  75.826

In [3]:

# ── Compute composite engagement score per department ──
# Weights: Performance 30%, KPI 25%, Task Completion 25%, Attendance 20%
weights = {'Performance Score': 0.30, 'KPI Score': 0.25, 
           'Task Completion (%)': 0.25, 'Attendance (%)': 0.20}

dept_mean = ee.groupby('Department')[list(weights.keys())].mean()
# Normalize each column to [0, 1] for fair weighting
dept_normalized = (dept_mean - dept_mean.min()) / (dept_mean.max() - dept_mean.min() + 1e-9)

dept_engagement_score = pd.DataFrame({
    'Department': dept_mean.index,
    'Composite_Engagement_Score': (dept_normalized * list(weights.values())).sum(axis=1).values
}).sort_values('Composite_Engagement_Score', ascending=False)

print("=== Composite Department Engagement Score ===")
print(dept_engagement_score.to_string(index=False))


=== Composite Department Engagement Score ===
Department  Composite_Engagement_Score
        Hr                    0.808661
   Finance                    0.566980
        It                    0.515837
     Sales                    0.429514
 Marketing                    0.072310


In [4]:

# ── Lowest-scoring employees list (bottom 10% by composite score) ──
ee['_composite'] = (
    0.30 * (ee['Performance Score'] / ee['Performance Score'].max()) +
    0.25 * (ee['KPI Score'] / ee['KPI Score'].max()) +
    0.25 * (ee['Task Completion (%)'] / ee['Task Completion (%)'].max()) +
    0.20 * (ee['Attendance (%)'] / ee['Attendance (%)'].max())
)
threshold = ee['_composite'].quantile(0.10)
low_performers = ee[ee['_composite'] <= threshold][
    ['Employee ID', 'Name', 'Department', 'Job Role', '_composite',
     'Performance Score', 'KPI Score', 'Attendance (%)']
].sort_values('_composite').head(20)

print(f"\n=== Bottom 10% Employees (composite score <= {threshold:.3f}) ===")
print(low_performers.to_string(index=False))



=== Bottom 10% Employees (composite score <= 0.741) ===
 Employee ID                Name Department                 Job Role  _composite  Performance Score  KPI Score  Attendance (%)
      374439 Zaina Bhattacharyya         Hr   Recruitment Specialist    0.647085                 50      63.52           75.13
      509441  Tushar Ranganathan      Sales     Business Development    0.649685                 50      61.66           80.14
      761004        Himmat Walla  Marketing       Content Strategist    0.659718                 53      65.51           75.84
      991989        Saira Kalita         Hr   Recruitment Specialist    0.667179                 56      61.31           77.76
      149390        Dishani Kara         Hr       Employee Relations    0.668009                 51      62.64           83.20
      539556    Ryan Chakrabarti         It             Data Analyst    0.669641                 51      64.81           78.16
      364851          Krish Sama         Hr   Recruitm

In [5]:

# ── Promotion eligibility breakdown by department ──
if 'Promotion Eligibility' in ee.columns:
    promo = ee.groupby(['Department','Promotion Eligibility']).size().unstack(fill_value=0)
    print("\n=== Promotion Eligibility by Department ===")
    print(promo)



=== Promotion Eligibility by Department ===
Promotion Eligibility   No  Yes
Department                     
Finance                872  144
Hr                     856  154
It                     845  129
Marketing              835  130
Sales                  897  138


In [6]:

# ── Save processed engagement analytics ──
dept_agg.to_csv(f'{PROC}/department_engagement_summary.csv', index=False)
dept_engagement_score.to_csv(f'{PROC}/department_composite_scores.csv', index=False)
low_performers.to_csv(f'{PROC}/low_performers.csv', index=False)
print("Saved: department_engagement_summary.csv, department_composite_scores.csv, low_performers.csv")


Saved: department_engagement_summary.csv, department_composite_scores.csv, low_performers.csv


**Engagement analytics complete.** Department-level aggregations only. No per-employee join with attrition data (different populations). Composite score and bottom-10% list saved.